# Data Quality Assessment


## Objective

The objective of this phase is to assess the quality, consistency, and reliability of the raw retail transaction data before it is used for ETL, data modeling, and downstream analysis.

The assessment focuses on identifying missing values, duplicate records, data type issues, invalid dates, relationship inconsistencies, invalid numeric values, and statistical outliers. The findings will be used to determine whether any data cleaning or transformation is required before proceeding to the ETL workflow.


## Data Quality Checklist

The following checks will be performed to assess the quality and consistency of the raw dataset:

| Data Quality Dimension          | Assessment                                                                             |
| ------------------------------- | -------------------------------------------------------------------------------------- |
| Completeness                    | Check for missing values                                                               |
| Uniqueness                      | Check for exact duplicate records and understand identifier repetition                 |
| Data Type Validity              | Verify that columns have appropriate data types                                        |
| Date Validity                   | Check for invalid or unparseable dates and date coverage                               |
| Entity Relationship Consistency | Validate relationships between transactions, customers, SKUs, and categories           |
| Quantity Validity               | Check for zero, negative, fractional, and extreme quantities                           |
| Sales Amount Validity           | Check for zero, negative, and extreme sales values                                     |
| Outlier Analysis                | Identify and assess statistically unusual observations                                 |
| Data Quality Decisions          | Document required cleaning actions and determine which observations should be retained |


In [2]:
import pandas as pd

In [3]:
from pathlib import Path
print(Path.cwd())

c:\Users\ASUS\Desktop\Data science\lol\project 2\Retail-Revenue-Intelligence\notebooks


## Load Dataset

In [4]:
df = pd.read_csv("../data/raw/scanner_data.csv")

## Missing Values

In [5]:
df.isnull().sum()

Unnamed: 0        0
Date              0
Customer_ID       0
Transaction_ID    0
SKU_Category      0
SKU               0
Quantity          0
Sales_Amount      0
dtype: int64

In [6]:
df.isnull().sum().sum()

0

The dataset contains no missing values. All 8 columns have 131,706 non-null values, matching the total number of records in the dataset.

Finding: No missing-value treatment is required.

## Duplicate Records

In [7]:
df.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
131701    False
131702    False
131703    False
131704    False
131705    False
Length: 131706, dtype: bool

In [8]:
df['Transaction_ID'].duplicated().sum()

67024

The dataset contains 0 exact duplicate rows, indicating that no records are completely duplicated across all columns.

Transaction_ID is not expected to be unique at the row level because the dataset is recorded at the transaction line-item level. A single transaction may contain multiple SKUs, resulting in multiple rows with the same Transaction_ID.Therefore, repeated Transaction_ID values are expected and are not treated as duplicate records.

Finding

No exact duplicate records were identified. Repeated Transaction_ID values are expected due to the line-item structure of the dataset

## Dataset Grain / Transaction Structure

In [9]:
# How many rows each transaction contains
df.groupby('Transaction_ID').size().value_counts().sort_index()

1     32906
2     16845
3      6802
4      3478
5      1851
6      1036
7       646
8       426
9       254
10      167
11       94
12       64
13       52
14       23
15       16
16        4
17        6
18        4
19        5
20        1
21        2
Name: count, dtype: int64

In [10]:
# transactions containing multiple line items
df.groupby('Transaction_ID').size().sort_values(ascending=False).head(10)

Transaction_ID
1348     21
19098    21
42295    20
34204    19
16271    19
34320    19
28731    19
39032    19
3181     18
27472    18
dtype: int64

In [11]:
# Inspect one transaction with multiple rows
df[df['Transaction_ID'] == 76]

,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
125,126,02/01/2016,4122,76,SJS,4BPPU,0.67,118.35
126,127,02/01/2016,4122,76,29A,LPLTZ,1.00,3.90
127,128,02/01/2016,4122,76,N8U,TL4WZ,1.00,4.00
128,129,02/01/2016,4122,76,9ZX,PV9XZ,4.00,33.60
129,130,02/01/2016,4122,76,OXH,MM77N,2.00,5.37
130,131,02/01/2016,4122,76,SJS,U4XSE,2.00,11.02
131,132,02/01/2016,4122,76,FU5,NOKHJ,2.00,22.26
132,133,02/01/2016,4122,76,LPF,9F8O1,2.00,12.36
133,134,02/01/2016,4122,76,IEV,NZQS5,2.00,12.27
134,135,02/01/2016,4122,76,LPF,GYQNP,2.00,14.40


The dataset is at the transaction-line-item level, not the transaction level. A single Transaction_ID can appear across multiple rows because one transaction can contain multiple SKU line items.

## Data Type Validation

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      131706 non-null  int64  
 1   Date            131706 non-null  object 
 2   Customer_ID     131706 non-null  int64  
 3   Transaction_ID  131706 non-null  int64  
 4   SKU_Category    131706 non-null  object 
 5   SKU             131706 non-null  object 
 6   Quantity        131706 non-null  float64
 7   Sales_Amount    131706 non-null  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 8.0+ MB


In [13]:
df['Unnamed: 0'].head()

0    1
1    2
2    3
3    4
4    5
Name: Unnamed: 0, dtype: int64

In [34]:
(df['Unnamed: 0']== df.index+1).all()

True

Data Type Validity: Most columns have appropriate data types for their intended use. The Date column is stored as an object and will be converted to datetime during the ETL process to support time-based analysis. The Unnamed: 0 column is a sequential index-like column with no business meaning and will be removed during ETL

**Finding:**

 Date requires conversion to datetime, and Unnamed: 0 will be removed during ETL. No other immediate data type issues were identified.

## Date Validation

In [14]:
date_check = pd.to_datetime(
    df['Date'],
    dayfirst=True,
    errors='coerce'
)

In [15]:
print('invalid dates:', date_check.isna().sum())
print('start_date', date_check.min())
print('last_date', date_check.max())

invalid dates: 0
start_date 2016-01-02 00:00:00
last_date 2016-12-31 00:00:00


All transaction dates were successfully parsed, with 0 invalid dates identified. The dataset covers transactions from January 2, 2016, through December 31, 2016.

Finding: No invalid dates were identified. The dataset covers nearly the full 2016 calendar year.

## Entity Relationship Consistency

In [16]:
df.groupby('Transaction_ID')['Customer_ID'].nunique().value_counts()

Customer_ID
1    64682
Name: count, dtype: int64

In [17]:
df.groupby('Transaction_ID')['Date'].nunique().value_counts()

Date
1    64682
Name: count, dtype: int64

In [18]:
df.groupby('SKU')['SKU_Category'].nunique().value_counts()

SKU_Category
1    5242
Name: count, dtype: int64

| Relationship           | Expected                                
| ---------------------- | --------------------------------------- 
| Transaction → Customer | One transaction belongs to one customer 
| Transaction → Date     | One transaction has one date            
| SKU → Category         | One SKU belongs to one category         


The relationships between key entities were checked for consistency. Each transaction is associated with exactly one customer and one date, while each SKU is associated with exactly one SKU category. No inconsistencies were identified.

Finding: The key relationships in the dataset are consistent.

## Quantity Validation

In [19]:
df['Quantity'].describe()

count    131706.000000
mean          1.485311
std           3.872667
min           0.010000
25%           1.000000
50%           1.000000
75%           1.000000
max         400.000000
Name: Quantity, dtype: float64

In [20]:
print('Zeroes :', (df['Quantity'] == 0).sum())
print('Negatives :', (df['Quantity'] < 0).sum())
print("Fractional quantities:", (df['Quantity'] % 1 != 0).sum())

Zeroes : 0
Negatives : 0
Fractional quantities: 375


In [21]:
fractional_qty = df[df['Quantity'] % 1 != 0]
fractional_qty[['SKU_Category', 'SKU', 'Quantity', 'Sales_Amount']].head(20)

,SKU_Category,SKU,Quantity,Sales_Amount
125,SJS,4BPPU,0.67,118.35
1111,PBV,8OXUS,0.34,13.44
1289,C3Y,OUWHY,0.25,2.08
1290,C3Y,RMOC0,0.25,2.19
1504,XG4,JSLMY,0.10,4.05
1642,LRR,U8PIH,0.34,18.52
1836,LRR,U8PIH,0.33,17.97
2053,MTR,FBVED,0.50,3.85
2811,MTR,FBVED,0.50,3.85
3016,PBV,XKX92,0.33,16.04


In [22]:
fractional_qty['SKU_Category'].value_counts().head(10)

SKU_Category
PBV    154
C3Y     55
9ZX     38
LPF     16
ZJG     15
LRR     14
MTR     12
SJS      8
J4R      7
ARZ      7
Name: count, dtype: int64

In [23]:
df[df['Quantity'] == df['Quantity'].max()]

,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
77676,77677,07/09/2016,19791,42595,H8O,CKDW0,400.0,16.37



The `Quantity` column was assessed for zero, negative, fractional, and extreme values.

No zero or negative quantities were identified, meaning all 131,706 records contain positive quantities. The median quantity is 1, while the maximum quantity is 400, indicating the presence of potentially extreme values.

A total of 375 records (0.28%) contain fractional quantities. These observations were retained because fractional-unit sales may represent valid business transactions. The maximum quantity of 400 was also reviewed as a potential extreme value but was not confirmed to be a data-quality error.

**Finding:** No invalid zero or negative quantities were identified. Fractional and extreme quantities were investigated but were not confirmed to be data-quality errors. Therefore, no quantity records were removed.


## Sales Amount Validation

In [24]:
df['Sales_Amount'].describe()

count    131706.000000
mean         11.981524
std          19.359699
min           0.020000
25%           4.230000
50%           6.920000
75%          12.330000
max         707.730000
Name: Sales_Amount, dtype: float64

In [25]:
print('zero sales:', (df['Sales_Amount'] == 0).sum())
print('negative sales:', (df['Sales_Amount'] < 0).sum())

zero sales: 0
negative sales: 0


No clearly invalid sales amounts were identified. There are no zero or negative sales values. However, the distribution is right-skewed, with a maximum sales amount of $707.73 compared with a median of $6.92. High-value observations should be investigated as potential outliers but should not be removed without business justification.

Finding: Sales_Amount appears valid at a basic data-quality level, but extreme high-value observations require further investigation.

In [26]:
df.nlargest(10, 'Sales_Amount')[
    ['Date', 'Customer_ID', 'Transaction_ID',
     'SKU_Category', 'SKU', 'Quantity', 'Sales_Amount']
]

,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
104910,09/11/2016,20906,54485,6BZ,09K06,6.0,707.73
6059,19/01/2016,232,2874,6BZ,8CAJ4,5.0,531.27
69200,12/08/2016,4542,38844,PBV,BR7EP,2.0,505.59
102777,17/07/2016,20906,35034,6BZ,09K06,4.0,471.83
42261,29/04/2016,10169,20852,6BZ,NACZN,3.0,424.63
25799,16/03/2016,4230,12497,6BZ,P35XJ,3.0,408.53
126780,27/12/2016,21065,63978,A38,RYHT4,2.0,398.13
98423,22/10/2016,18726,51560,SJS,OXKQ8,2.0,362.77
33515,06/04/2016,17581,16429,HTR,S3KHW,3.0,357.75
54846,01/06/2016,20906,27258,6BZ,09K06,3.0,353.87


In [27]:
df.nsmallest(10, 'Sales_Amount')[
    ['Date', 'Customer_ID', 'Transaction_ID',
     'SKU_Category', 'SKU', 'Quantity', 'Sales_Amount']
]

,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
42620,01/05/2016,17093,21066,2ML,3483Q,1.0,0.02
44196,05/05/2016,13219,21860,2ML,3483Q,1.0,0.02
45095,07/05/2016,4515,22332,2ML,3483Q,1.0,0.02
45255,08/05/2016,215,22420,2ML,3483Q,1.0,0.02
45736,09/05/2016,5975,22656,2ML,3483Q,1.0,0.02
46474,11/05/2016,22111,23014,2ML,3483Q,1.0,0.02
46672,12/05/2016,12078,23118,2ML,3483Q,1.0,0.02
46872,12/05/2016,18699,23217,2ML,3483Q,1.0,0.02
47396,13/05/2016,974,23464,2ML,3483Q,1.0,0.02
47562,13/05/2016,10030,23548,2ML,3483Q,1.0,0.02


Sales Amount Validity: No zero or negative sales amounts were identified. Sales amounts range from $0.02 to $707.73, with a median of $6.92. Investigation of extreme values indicates that both high and low sales amounts are associated with recurring SKUs and categories rather than isolated records. The minimum value of $0.02 repeatedly occurs for SKU 3483Q, while several high-value observations are concentrated in specific SKUs/categories. Therefore, no sales amount records were removed based solely on their extreme values.

Finding: No clearly invalid Sales_Amount values were identified. Extreme values will be retained and considered during downstream analysis.

## Outlier Analysis

In [28]:
# quantity outlier analysis

Q1_quantity = df['Quantity'].quantile(0.25)
Q3_quantity = df['Quantity'].quantile(0.75)
IQR_quantity = Q3_quantity - Q1_quantity

lower_quantity = Q1_quantity - 1.5 * IQR_quantity
upper_quantity = Q3_quantity + 1.5 * IQR_quantity

# sales amount outlier analysis

Q1_sales = df['Sales_Amount'].quantile(0.25)
Q3_sales = df['Sales_Amount'].quantile(0.75)
IQR_sales = Q3_sales - Q1_sales

lower_sales = Q1_sales - 1.5 * IQR_sales
upper_sales = Q3_sales + 1.5 * IQR_sales

print('Quantity')
print('Q1 :',Q1_quantity)
print('Q3 :',Q3_quantity)
print('IQR :',IQR_quantity)
print('Lower Bound :',lower_quantity)
print('Upper Bound :',upper_quantity)

print('\nSales Amount')
print('Q1 :',Q1_sales)
print('Q3 :',Q3_sales)
print('IQR :',IQR_sales)
print('Lower Bound :',lower_sales)
print('Upper Bound :',upper_sales)

Quantity
Q1 : 1.0
Q3 : 1.0
IQR : 0.0
Lower Bound : 1.0
Upper Bound : 1.0

Sales Amount
Q1 : 4.23
Q3 : 12.33
IQR : 8.1
Lower Bound : -7.919999999999998
Upper Bound : 24.479999999999997


In [29]:
#outliers count

quantity_outliers = df[(df['Quantity'] < lower_quantity) | (df['Quantity'] > upper_quantity)]
sales_outliers = df[(df['Sales_Amount'] < lower_sales) | (df['Sales_Amount'] > upper_sales)]

print('Quantity_outliers count:', quantity_outliers.shape[0])
print('Sales_outliers count:', sales_outliers.shape[0])


Quantity_outliers count: 28965
Sales_outliers count: 12713


In [30]:
# outlier percentage

print('Quantity_outlier_percentage:', round(quantity_outliers.shape[0]/df.shape[0]* 100, 2))
print('Sales_outlier_percentage:', round(sales_outliers.shape[0]/df.shape[0]* 100, 2))

Quantity_outlier_percentage: 21.99
Sales_outlier_percentage: 9.65


In [31]:
quantity_outliers.head(20)

,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
20,21,02/01/2016,3874,15,SFC,26XOE,2.00,4.75
42,43,02/01/2016,3071,26,N8U,UNJKW,2.00,4.23
48,49,02/01/2016,6459,31,JKC,0TLU5,2.00,2.13
74,75,02/01/2016,1098,47,EMC,23Y9E,3.00,15.18
93,94,02/01/2016,473,58,J4R,AODV3,2.00,8.73
94,95,02/01/2016,473,58,A38,MZL43,3.00,8.14
99,100,02/01/2016,8537,63,AEB,4EKR9,2.00,17.17
107,108,02/01/2016,3992,64,2O0,UCK6F,2.00,11.18
125,126,02/01/2016,4122,76,SJS,4BPPU,0.67,118.35
128,129,02/01/2016,4122,76,9ZX,PV9XZ,4.00,33.60


In [32]:
sales_outliers.head(20)

,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
60,61,02/01/2016,1655,39,OBD,RZW8B,1.00,26.57
70,71,02/01/2016,824,45,Z4O,GID11,1.00,30.00
125,126,02/01/2016,4122,76,SJS,4BPPU,0.67,118.35
128,129,02/01/2016,4122,76,9ZX,PV9XZ,4.00,33.60
145,146,02/01/2016,178,84,R6E,HO1M5,1.00,24.93
146,147,02/01/2016,178,84,Q4N,699MN,1.00,34.06
201,202,02/01/2016,7443,113,C3R,BP4WV,1.00,25.62
209,210,02/01/2016,1659,116,DJI,L81FM,1.00,25.16
338,339,03/01/2016,9157,199,DJI,OQCLK,1.00,30.87
364,365,04/01/2016,8858,212,9ZX,LJ26I,2.00,43.30


### Outlier Analysis

The IQR method was applied to assess unusually high or low values in `Quantity` and `Sales_Amount`.

* **Quantity:** The IQR method flags 21.99% of observations as outliers because Q1 and Q3 are both 1, resulting in an IQR of 0. This threshold is not informative for this highly concentrated distribution. The flagged values therefore require business-context interpretation rather than automatic removal. No clearly invalid quantity values were identified. The observations will be retained for downstream analysis.

* **Sales Amount:** The IQR method identified values above approximately 24.48 as statistically unusual, representing 9.65% of observations. A sample review of the flagged transactions did not indicate obvious data-entry errors. The extreme values were observed across recurring SKUs and categories, suggesting that they may represent legitimate  differences in transaction value rather than isolated data-entry errors.

* **Data Quality Decision:** Statistical outliers were not automatically treated as invalid records. Since the flagged values appear to represent legitimate transaction behavior rather than data-quality issues, all observations were retained for subsequent analysis.


## Data Quality Summary & Cleaning Decisions

The dataset was assessed across completeness, duplicates, date validity, identifiers, numeric fields, and statistical outliers.

| Check                 | Finding                                                                                                                                          | Decision                |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------ | ----------------------- |
| Missing Values        | No missing values identified across the dataset                                                                                                  | No action required      |
| Duplicate Rows        | 0 exact duplicate rows                                                                                                                           | No action required      |
| Transaction IDs       | `Transaction_ID` is repeated across multiple rows, indicating transaction-level line items rather than duplicate records                         | Retain                  |
| Date Validity         | No invalid dates; coverage from January 2, 2016 to December 31, 2016                                                                             | Retain                  |
| Quantity              | No zero or negative quantities; all values are positive                                                                                          | Retain                  |
| Sales Amount          | No zero or negative sales amounts; all values are positive                                                                                       | Retain                  |
| Quantity Outliers     | IQR method flagged 21.99%; however, Q1 = Q3 = 1, resulting in IQR = 0 and making the method unsuitable for meaningful business outlier detection | Retain                  |
| Sales Amount Outliers | IQR method flagged 9.65% as statistically unusual; review did not indicate obvious data-entry errors                                             | Retain                  |
| Overall Decision      | No confirmed data-quality errors requiring row-level deletion were identified                                                                    | Retain all observations |

### Final Data Quality Decision

The dataset is considered suitable for downstream analysis. No rows were removed because the identified unusual values appear to represent legitimate transaction behavior rather than confirmed data-quality errors. The original data will be retained, with appropriate transformations and structural cleaning performed during the ETL stage.


Planned ETL actions:

1. Remove the source index column (`Unnamed: 0`).
2. Convert `Date` from object to datetime.
3. Preserve the transaction-line-item grain.
4. Retain fractional quantities.
5. Retain statistically unusual quantities and sales amounts; no confirmed data-quality errors were identified.
6. Preserve validated entity relationships.
7. Prepare the cleaned data for downstream dimensional modeling.